# Phase 0 finish + phase 1 pilot### study `capacity_axis_20260902` — the E3 capacity sweepRun the cells in order. **Cell 1 decides what is feasible** — on a T4 the pilot is ~6 h and sitsright on the approval threshold; on an L4 it is ~1.7 h.**The main sweep is not run here.** 42 shards is ~21 h on L4 and ~78 h on T4, against ~16 h onCheaha's A100s where job arrays run it unattended. This notebook finishes the harness and the pilot;the pilot's measured cost then decides where the sweep goes.

## 1 · Which GPU, and what fits

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
mult = 1.0 if "A100" in name else 2.6 if "L4" in name else 9.4 if "T4" in name else 3.0
A = 16.0  # gpt2-124M seconds per target on an A100, ~15% MFU

def shard(k, n):  # one (k, seed) shard = n persons/arm x 2 arms x 2 fields attacks
    return 0.01*mult if k == 0 else A*mult*((k+10)/30)*n*4/3600

probe = sum(shard(k, 3)  for k in (1, 20, 64))
pilot = sum(shard(k, 25) for k in (0, 1, 20))
repro = shard(20, 25)
print(f"\nGPU: {name}   (~{mult:.1f}x an A100 for this workload)\n")
print(f"  cost probe  n=3, k=1/20/64 : {probe:5.2f} h")
print(f"  pilot       k=0/1/20 n=25  : {pilot:5.2f} h")
print(f"  repro       k=20           : {repro:5.2f} h")
print(f"  ---------------------------------------")
print(f"  this notebook              : {probe+pilot+repro:5.2f} h")
print(f"\n  (main sweep, 42 shards    : {sum(shard(k,25) for k in [1,2,3,4,6,8,12,16,20,24,32,48,64])*3 + shard(0,25)*3:5.1f} h -- NOT run here)")
if mult > 5:
    print("\n  T4: ~12 h -- a single session will not finish this and it exceeds")
    print("  the study's confirm_above of 6 accelerator-hours. Switch to L4, or")
    print("  run only the cost probe + k=0 + k=1 here and take the rest to Cheaha.")
elif mult > 1.5:
    print("\n  L4: ~3.3 h. Comfortable in one session, and under confirm_above.")

## 2 · Drive, and the repo **on** Drive`config.py` derives `data/`, `models/` and `results/` from the repo root with **no env override**, sothe repo has to live on Drive. Clone it under `/content` and a reclaimed session takes the corpus andthe checkpoint with it.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
%cd /content/drive/MyDrive
!git clone https://github.com/jackyluo-learning/PII_Extraction.git 2>/dev/null || echo "already cloned"
%cd /content/drive/MyDrive/PII_Extraction
!git pull --ff-only
!git log --oneline -3

**Check the three SHAs before going on.** `71a8952`, `d561c48`, `780ee1c` carry the eightphase-0 gates. Without them the sweep would still take `trained[:25]` — which contains **no f=20people at all** — write no manifest, and let C4 into the corpus silently.

In [ ]:
import subprocess
have = subprocess.run(["git","log","--format=%h","-20"], capture_output=True, text=True).stdout.split()
missing = [s for s in ("71a8952","d561c48","780ee1c") if s not in have]
assert not missing, f"MISSING phase-0 gates: {missing} -- push them from the laptop first, do not run"
print("phase-0 gates present")

## 3 · Dependencies

In [ ]:
!pip install -q -r requirements.txt
import faker, torch, lifelines
print("faker", faker.VERSION, "| torch", torch.__version__, "| lifelines", lifelines.__version__)
assert lifelines.__version__ == "0.30.0", "lifelines is pinned exactly: its interval-censoring API moved between releases and a change could flip H4"

## 4 · t0-8a · Corpus, with both new halts armedTwo assertions fire here rather than being left to memory:* **C4 halt** — raises if any Common-Crawl passage was used. C4 carries real names and emails, which  would break this corpus's "no real personal data" guarantee.* **Faker disjointness** — raises if any SSN or email collides between the trained pool (`seed`) and  the control pool (`seed+1000`). One collision moves a control record's true membership and inflates  the forcing floor.If either raises, **stop**. Do not work around it.

In [ ]:
%env PII_N_CONTROLS=50
%env PII_DEVICE_PROFILE=auto
!python data_generation.py

In [ ]:
import json
m = json.load(open("data/corpus_metadata.json"))
sc = m["public_passages"]["source_counts"]
print("public-passage sources:", sc)
assert sc.get("c4", 0) == 0, "C4 contributed -- real PII may be present, regenerate"
print("clean: no Common-Crawl passages")

## 5 · t0-8b · Retrain gpt2-124M

In [ ]:
%env PII_DEVICE_PROFILE=auto
!python train.py --model gpt2

In [ ]:
import json
print(json.load(open("models/gpt2/train_meta.json"))["pii_eval_losses"][-3:])
# expect the PII eval loss near zero, as in run2

## 6 · t1-2 · Cost probe — the cheap way to answer itThe question is how per-attack cost scales with `k`, and that needs a few attacks at each extreme,not a hundred. `_get_top_candidates` builds `k·B = 256k` candidate tensors per iteration in a nestedPython loop and keeps only 512 — at `k=64` that is 16,384 built to keep 512 — so the overhead growsfaster than the FLOP term and a single mid-grid point cannot reveal it.`PII_CAP_SWEEP_N=3` and a **separate `run_id`**, so this never mixes into the evidence.

In [ ]:
%env PII_RUN_ID=e3a_cost
%env PII_CAP_SWEEP_N=3
%env PII_GCG_ITERS=200
%env PII_FIELDS=ssn,email
%env PII_DEVICE_PROFILE=auto
import os, time, subprocess
for k in (1, 20, 64):
    os.environ["PII_CAP_K"] = str(k); t0 = time.time()
    subprocess.run(["python","experiments.py","--exp","E3","--model","gpt2","--seed","42"], check=True)
    print(f"  >>> k={k}: {time.time()-t0:.0f} s wall-clock for 6 attacks")

In [ ]:
import glob, pandas as pd
d = pd.concat([pd.read_parquet(p) for p in glob.glob("results/attempts/e3a_cost__*.parquet")])
s = d.groupby("capacity_k").wallclock_s.mean()
print(s.round(1).to_string())
base = s.get(20)
print("\nper-attack seconds, normalised to k=20:")
for k, v in s.items():
    linear = (k+10)/30
    print(f"  k={k:2.0f}  measured {v/base:5.2f}x   linear-in-(k+T) predicts {linear:5.2f}x"
          f"   {'<-- overhead term is real' if v/base > linear*1.25 else ''}")

## 7 · t1-1 · The sanity gate — run k=0 and read it before anything else`k=0` has no free tokens, so there is nothing to optimise and the attack degenerates to a naturalprompt. `α_0` **must** be ~0 — run2's own gpt2 fixed-probe EMR was 0.0. A non-trivial value means`exact_match` is firing on something it should not, and every `k` above it would be offset by thaterror.

In [ ]:
%env PII_RUN_ID=e3a
%env PII_CAP_SWEEP_N=25
%env PII_CAP_K=0
!python experiments.py --exp E3 --model gpt2 --seed 42

In [ ]:
import glob, pandas as pd
a0 = pd.concat([pd.read_parquet(p) for p in glob.glob("results/attempts/e3a__*_k0.parquet")])
c = a0[a0.target_membership == "control"]
print(f"alpha_0 = {c.exact_match.mean():.4f}   n={len(c)} control targets")
print(f"EMR(D)  = {a0[a0.target_membership=='trained'].exact_match.mean():.4f}")
assert c.exact_match.mean() < 0.02, "SANITY GATE FAILED -- exact_match fires with zero capacity. The study BLOCKS."
print("\ngate passed")

## 8 · Pilot shards at full n — one cell each, so you can stop between them

In [ ]:
%env PII_CAP_K=1
!python experiments.py --exp E3 --model gpt2 --seed 42

In [ ]:
%env PII_CAP_K=20
!python experiments.py --exp E3 --model gpt2 --seed 42

## 9 · t1-3 · Arm sizes, tier composition, and the cross-shard invariants`tier_composition` should read `{'1': 3, '5': 7, '20': 15}`. Before `d561c48` the prefix gave`{'1': 10, '5': 15}` — **no f=20 at all**, the tier that is 60% of the trained population and themost memorised.`compare()` must report `ok: True`. If `target_subset_hash` differs between shards the paired designacross `k` is gone — **block; do not analyse the shards that agree.**

In [ ]:
import glob, json, run_manifest
for p in sorted(glob.glob("results/manifests/e3a__*.json")):
    d = json.load(open(p))
    print(f"  k={str(d['shard']['capacity_k']):<3} arms={d['arm_sizes']} "
          f"tiers={d['tier_composition']} subset={d['target_subset_hash']} N={d['gcg_iters']}")
r = run_manifest.compare(sorted(glob.glob("results/manifests/e3a__*.json")))
print("\ninvariants:", {k: r[k] for k in ("ok","n_shards","distinct")})
assert r["ok"], "target_subset_hash or gcg_iters differ across shards -- BLOCK"
d = json.load(open(sorted(glob.glob("results/manifests/e3a__*.json"))[0]))
assert "20" in d["tier_composition"], "f=20 tier missing -- the stratified-selection gate did not land"
print("\nf=20 tier present; invariants hold")

## 10 · t1-4 · Reproducibility check**Restart the runtime first** (Runtime → Disconnect and delete runtime), then re-run cells 2 and 3.A same-session re-run cannot detect session-to-session nondeterminism, which is the thing beingtested.

In [ ]:
%env PII_RUN_ID=e3a_repro
%env PII_CAP_SWEEP_N=25
%env PII_GCG_ITERS=200
%env PII_FIELDS=ssn,email
%env PII_DEVICE_PROFILE=auto
%env PII_CAP_K=20
!python experiments.py --exp E3 --model gpt2 --seed 42

In [ ]:
import glob, pandas as pd
a = pd.concat([pd.read_parquet(p) for p in glob.glob("results/attempts/e3a__*_k20.parquet")])
b = pd.concat([pd.read_parquet(p) for p in glob.glob("results/attempts/e3a_repro__*_k20.parquet")])
j = a.merge(b, on=["person_id","field"], suffixes=("_a","_b"))
print(f"{'arm':10s} {'n':>4} {'p':>7} {'flip':>7} {'2p(1-p)':>9}")
for arm, g in j.groupby("target_membership_a"):
    p = g.exact_match_a.mean(); flip = (g.exact_match_a != g.exact_match_b).mean()
    print(f"{arm:10s} {len(g):4d} {p:7.3f} {flip:7.3f} {2*p*(1-p):9.3f}")

The comparator is `2p(1−p)` — the disagreement expected under pure stochastic re-draw at anunchanged true rate — **per arm**, never pooled. An earlier draft used `1−EMR`, which is the wrongnull.---## 11 · What to bring homeEverything is already on Drive. The evidence is small:

In [ ]:
!du -sh results/attempts results/manifests data/corpus models/gpt2 2>/dev/null
!ls -1 results/attempts results/manifests

**Pull `results/attempts/*.parquet` and `results/manifests/*.json` to the laptop.** They are theevidence; the checkpoint stays on Drive.### If a session dies mid-shardNothing is lost beyond the person in flight — the manifest is written before the first attack and thelog flushes per person. Re-run that one `PII_CAP_K`; the parquet is rewritten whole, so a partialfile is simply replaced.### ThenThe cost probe's measured scaling decides where the 42-shard main sweep runs. At ~21 h on L4 against~16 h unattended on Cheaha's A100 job arrays, Cheaha is the likely answer.